Q1

In [1]:
class UtilityBasedAgent:
    def __init__(self, hospital_graph, start_ward):
        self.hospital = hospital_graph
        self.current_ward = start_ward
        self.moves = 0
        self.critical_treated = 0
        # Count total critical patients initially
        self.total_critical = sum(1 for w in self.hospital.values() if w['status'] == 'Critical')

    def utility(self):
        # Utility formula given in the exam: (10 * critical patients treated - moves)
        return (10 * self.critical_treated) - self.moves

    def dfs_find_critical(self, start, visited=None, path=None):
        """Finds a path to the next critical patient using DFS."""
        if visited is None:
            visited = set()
        if path is None:
            path = []

        visited.add(start)
        path = path + [start]

        # If we found a critical ward that isn't our starting point (unless we just started)
        if self.hospital[start]['status'] == 'Critical' and len(path) > 1:
            return path

        for neighbor in self.hospital[start]['adjacent']:
            if neighbor not in visited:
                result = self.dfs_find_critical(neighbor, visited, path)
                if result:
                    return result
        return None

    def run_simulation(self):
        print(f"--- Simulation Started at {self.current_ward} ---")

        # Check if starting ward is already critical
        if self.hospital[self.current_ward]['status'] == 'Critical':
            print(f"Action: Delivered medicine at {self.current_ward} (Initial)")
            self.hospital[self.current_ward]['status'] = 'Normal'
            self.critical_treated += 1
            print(f"Utility: {self.utility()}")

        while self.critical_treated < self.total_critical:
            path = self.dfs_find_critical(self.current_ward)

            if not path:
                print("No reachable critical patients left.")
                break

            # Move along the DFS path
            for step in path[1:]: # Skip the first element as it's the current ward
                self.current_ward = step
                self.moves += 1
                print(f"Action: Moved to {self.current_ward}")

                # If we reach the target critical ward, deliver medicine
                if self.hospital[self.current_ward]['status'] == 'Critical':
                    print(f"Action: Delivered medicine at {self.current_ward}")
                    self.hospital[self.current_ward]['status'] = 'Normal'
                    self.critical_treated += 1

                print(f"Current Utility: {self.utility()}")

        print("\n--- Simulation Complete ---")
        print(f"Final Ward: {self.current_ward}")
        print(f"Total Moves: {self.moves}")
        print(f"Total Critical Treated: {self.critical_treated}")
        print(f"Final Utility: {self.utility()}")


# Environment Representation
hospital_env = {
    "WardA": {"status": "Critical", "adjacent": ["WardB", "WardC"]},
    "WardB": {"status": "Normal", "adjacent": ["WardA", "WardD"]},
    "WardC": {"status": "Critical", "adjacent": ["WardA", "WardD"]},
    "WardD": {"status": "Empty", "adjacent": ["WardB", "WardC"]},
}

# Run the agent
agent = UtilityBasedAgent(hospital_env, start_ward="WardD")
agent.run_simulation()

--- Simulation Started at WardD ---
Action: Moved to WardB
Current Utility: -1
Action: Moved to WardA
Action: Delivered medicine at WardA
Current Utility: 8
Action: Moved to WardB
Current Utility: 7
Action: Moved to WardD
Current Utility: 6
Action: Moved to WardC
Action: Delivered medicine at WardC
Current Utility: 15

--- Simulation Complete ---
Final Ward: WardC
Total Moves: 5
Total Critical Treated: 2
Final Utility: 15


Q2

In [2]:
import heapq

# Define the grid map
grid = [
    ['S', '.', '.', 'W', '.'],
    ['.', '#', '.', '#', '.'],
    ['.', '#', '.', '.', '.'],
    ['.', '.', 'F', '#', 'G'],
    ['.', '.', '.', '.', '.']
]

rows, cols = len(grid), len(grid[0])
costs = {'.': 1, 'W': 3, 'F': 5, 'S': 0, 'G': 1} # G treated as safe landing (cost 1)

# Find Start and Goal
start = goal = None
for r in range(rows):
    for c in range(cols):
        if grid[r][c] == 'S': start = (r, c)
        if grid[r][c] == 'G': goal = (r, c)

def manhattan_distance(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

def get_neighbors(r, c):
    neighbors = []
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)] # Up, Down, Left, Right
    for dr, dc in directions:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != '#':
            neighbors.append((nr, nc))
    return neighbors

def greedy_best_first_search(start, goal):
    # Priority queue stores tuples of (heuristic, current_node)
    pq = [(manhattan_distance(start, goal), start)]

    # Track the path (came_from) and total actual cost (g_score)
    came_from = {start: None}
    g_score = {start: 0}
    expanded_nodes = []

    while pq:
        # GBFS relies purely on the heuristic h(n)
        _, current = heapq.heappop(pq)
        expanded_nodes.append(current)

        if current == goal:
            break

        for neighbor in get_neighbors(*current):
            terrain = grid[neighbor[0]][neighbor[1]]
            move_cost = costs[terrain]
            new_g = g_score[current] + move_cost

            if neighbor not in came_from:
                g_score[neighbor] = new_g
                came_from[neighbor] = current
                h = manhattan_distance(neighbor, goal)
                heapq.heappush(pq, (h, neighbor))

    # Reconstruct path
    path = []
    curr = goal
    total_cost = g_score.get(goal, float('inf'))

    while curr is not None:
        path.append(curr)
        curr = came_from.get(curr)
    path.reverse()

    return expanded_nodes, path, total_cost

# Run GBFS
expanded, optimal_path, total_cost = greedy_best_first_search(start, goal)

print("Expanded Nodes Sequence:", expanded)
print("Optimal Path:", optimal_path)
print("Total Path Cost:", total_cost)

Expanded Nodes Sequence: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4)]
Optimal Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4)]
Total Path Cost: 9


Q3

In [3]:
import math
import random

# Cities and coordinates
cities = {
    0: (2, 3),
    1: (5, 7),
    2: (1, 9),
    3: (8, 2),
    4: (4, 5)
}
N = len(cities)

def calculate_distance(city1, city2):
    x1, y1 = cities[city1]
    x2, y2 = cities[city2]
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

def route_total_distance(route):
    total = 0
    # Include returning to the start city
    for i in range(len(route)):
        total += calculate_distance(route[i], route[(i + 1) % len(route)])
    return total

# 1. Roulette Wheel Selection
def roulette_wheel_selection(population, fitnesses):
    total_fitness = sum(fitnesses)
    pick = random.uniform(0, total_fitness)
    current = 0
    for i, fitness in enumerate(fitnesses):
        current += fitness
        if current > pick:
            return population[i]
    return population[-1]

# 2. Partially Mapped Crossover (PMX)
def pmx_crossover(parent1, parent2):
    size = len(parent1)
    p1, p2 = [0]*size, [0]*size

    # Initialize child arrays with None
    child1, child2 = [None]*size, [None]*size

    # Choose two random crossover points
    cxpoint1 = random.randint(0, size - 2)
    cxpoint2 = random.randint(cxpoint1 + 1, size - 1)

    # Copy the middle segment
    child1[cxpoint1:cxpoint2+1] = parent1[cxpoint1:cxpoint2+1]
    child2[cxpoint1:cxpoint2+1] = parent2[cxpoint1:cxpoint2+1]

    def map_elements(child, parent_main, parent_donor):
        # Map remaining elements
        for i in range(size):
            if not cxpoint1 <= i <= cxpoint2:
                val = parent_donor[i]
                while val in child:
                    idx = parent_main.index(val)
                    val = parent_donor[idx]
                child[i] = val
        return child

    child1 = map_elements(child1, parent1, parent2)
    child2 = map_elements(child2, parent2, parent1)

    return child1, child2

# 3. Inversion Mutation
def inversion_mutation(route, mutation_rate=0.1):
    if random.random() < mutation_rate:
        idx1, idx2 = sorted(random.sample(range(len(route)), 2))
        route[idx1:idx2+1] = reversed(route[idx1:idx2+1])
    return route

# Main Genetic Algorithm Loop
def genetic_algorithm(generations=100, pop_size=20):
    # Initialize random population
    population = [random.sample(range(N), N) for _ in range(pop_size)]

    best_route = None
    min_distance = float('inf')

    for gen in range(generations):
        # Calculate fitness (inverse of distance so smaller distance = higher fitness)
        distances = [route_total_distance(ind) for ind in population]
        fitnesses = [1.0 / d for d in distances]

        # Track best route
        current_min_idx = distances.index(min(distances))
        if distances[current_min_idx] < min_distance:
            min_distance = distances[current_min_idx]
            best_route = population[current_min_idx]

        new_population = []

        # Create new generation
        while len(new_population) < pop_size:
            p1 = roulette_wheel_selection(population, fitnesses)
            p2 = roulette_wheel_selection(population, fitnesses)

            c1, c2 = pmx_crossover(p1, p2)

            c1 = inversion_mutation(c1)
            c2 = inversion_mutation(c2)

            new_population.extend([c1, c2])

        population = new_population[:pop_size]

    return best_route, min_distance

# Run GA
best_route, final_distance = genetic_algorithm(generations=150, pop_size=30)
print("Output:")
print(f"Best Route: {best_route}")
print(f"Total Distance: {final_distance:.2f} units")

Output:
Best Route: [3, 0, 2, 1, 4]
Total Distance: 23.87 units


In [4]:
#Rank Based Selection
import random

def rank_based_selection(population, fitnesses):
    """
    Selects a parent using Rank-Based Selection.
    Assumes higher fitness values are better.
    """
    # Pair each individual with its fitness so we can sort them together
    paired = list(zip(population, fitnesses))

    # Sort the population by fitness in ascending order (worst to best)
    # Worst individual ends up at index 0 (Rank 1), Best at index N-1 (Rank N)
    paired.sort(key=lambda x: x[1])

    n = len(paired)
    total_ranks = n * (n + 1) / 2 # Sum of all ranks from 1 to N

    # Pick a random number between 0 and the total sum of ranks
    pick = random.uniform(0, total_ranks)

    current = 0
    # Iterate through the sorted population, assigning ranks 1 to N
    for rank, (individual, fitness) in enumerate(paired, start=1):
        current += rank
        if current >= pick:
            return individual

    # Fallback in case of floating point inaccuracies (returns the best individual)
    return paired[-1][0]
########################################################################################

#new code to adddddd
p1 = rank_based_selection(population, fitnesses)
p2 = rank_based_selection(population, fitnesses)